In [0]:
import sys
print(sys.version)

import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)

In [0]:
# Check if microsoft-aurora is already installed
import importlib.util

if importlib.util.find_spec("aurora") is None:
    print("Installing microsoft-aurora...")
    %pip install microsoft-aurora
else:
    print("microsoft-aurora is already installed, skipping installation.")

In [0]:
import os

checkpoint_path = "/dbfs/models/aurora-0.1-finetuned.ckpt"

if os.path.exists(checkpoint_path):
    print(f"Checkpoint file already exists at {checkpoint_path}, skipping download.")
else:
    print("Downloading checkpoint file...")
    # Create directory if it doesn't exist
    os.makedirs("/dbfs/models", exist_ok=True)
    !wget https://huggingface.co/microsoft/aurora/resolve/main/aurora-0.1-finetuned.ckpt -P /dbfs/models/
    print("Download complete.")

In [0]:
import os
import psutil

# Check available memory before loading
memory_info = psutil.virtual_memory()
available_gb = memory_info.available / (1024**3)
print(f"Available memory: {available_gb:.2f} GB")

if available_gb < 10:
    print("⚠️ Warning: Less than 10GB of available memory. Model loading may fail or be very slow.")
    print("Consider using a larger cluster or GPU-enabled compute.")

checkpoint_path = "/dbfs/models/aurora-0.1-finetuned.ckpt"

if not os.path.exists(checkpoint_path):
    print(f"Error: Checkpoint file not found at {checkpoint_path}")
    print("Please run the previous cell to download the checkpoint first.")
else:
    try:
        from aurora import AuroraHighRes
        print("Aurora HighRes model")
        model = AuroraHighRes()
        print("Loading model checkpoint...")
        model.load_checkpoint_local(checkpoint_path)
        print("Checkpoint loaded")
        model.eval()
        print("✓ Model loaded successfully")
    except Exception as e:
        print(f"Error loading model: {str(e)}")
        print("\nTroubleshooting tips:")
        print("1. Ensure you have sufficient memory (16GB+ recommended)")
        print("2. Consider using a GPU-enabled cluster for better performance")
        print("3. Close other notebooks or restart the cluster if memory is full")

In [0]:
%pip install cfgrib eccodes

In [0]:
%sh /Volumes/dlam-uat/utility/utility-volume/eccodes-init/eccodes_init.sh

In [0]:
# ecCodes patch for Databricks - run BEFORE importing cfgrib
import os

ECCODES_LIB = "/tmp/eccodes/lib/libeccodes.so"

if not os.path.exists(ECCODES_LIB):
    raise RuntimeError(
        f"ecCodes library not found at {ECCODES_LIB}. "
        "Ensure the init script ran successfully."
    )

os.environ['ECCODES_DIR'] = '/tmp/eccodes'
os.environ['LD_LIBRARY_PATH'] = f"/tmp/eccodes/lib:{os.environ.get('LD_LIBRARY_PATH', '')}"

import findlibs
_original_find = findlibs.find

def _patched_find(name, *args, **kwargs):
    if name == "eccodes":
        return ECCODES_LIB
    return _original_find(name, *args, **kwargs)

findlibs.find = _patched_find

print("ecCodes patch applied.")
 

In [0]:
import os
import xarray as xr
import numpy as np
import torch
import pickle
from aurora import AuroraHighRes, Batch, Metadata, rollout
from huggingface_hub import hf_hub_download
import warnings
warnings.filterwarnings("ignore")

# ------------------------------
# Configuration
# ------------------------------
merged_file = "/dbfs/tmp/ec_data_merged.grib"
output_dir  = "/dbfs/tmp/aurora_forecasts"

os.makedirs(output_dir, exist_ok=True)

In [0]:
# ------------------------------
# Helper functions
# ------------------------------
def fix_coordinates(ds):
    lon_values = ds.longitude.values.astype(np.float32)
    new_lons = np.mod(lon_values, 360)
    new_lons[new_lons == 360] = 0.0
    ds = ds.assign_coords(longitude=new_lons)
    ds = ds.sortby("longitude")
    return ds

def lon_360_to_180(lon):
    lon = lon.copy()
    lon[lon > 180] -= 360
    return lon

In [0]:
# ------------------------------
# Load static variables
# ------------------------------
print("Downloading static variables...")
static_path = hf_hub_download(
    repo_id="microsoft/aurora",
    filename="aurora-0.1-static.pickle"
)
with open(static_path, "rb") as f:
    static_vars = pickle.load(f)
    
print("Static variables loaded.")

In [0]:
# ------------------------------
# Load merged GRIB
# ------------------------------
print("Loading merged GRIB...")

sfc_ds = xr.load_dataset(
    merged_file,
    engine="cfgrib",
    filter_by_keys={"typeOfLevel": "surface"},
    decode_timedelta=True,
)

pl_ds = xr.load_dataset(
    merged_file,
    engine="cfgrib",
    filter_by_keys={"typeOfLevel": "isobaricInhPa"},
    decode_timedelta=True,
)

# Fix coordinates
sfc_ds = fix_coordinates(sfc_ds)
pl_ds  = fix_coordinates(pl_ds)

print("\n✓ All datasets loaded successfully")

In [0]:
# ------------------------------
# Prepare variables
# ------------------------------
# Surface — GRIB has no step dim, so values are [lat, lon]
# Aurora expects [B, T, H, W] → use [None, None] for both batch + time dims
sfc_ds_renamed = sfc_ds.rename({
    "t2m": "2t",
    "u10": "10u",
    "v10": "10v"
})

surf_vars = {
    "2t":  torch.from_numpy(sfc_ds_renamed["2t"].values[None, None]),
    "10u": torch.from_numpy(sfc_ds_renamed["10u"].values[None, None]),
    "10v": torch.from_numpy(sfc_ds_renamed["10v"].values[None, None]),
    "msl": torch.from_numpy(sfc_ds_renamed["msl"].values[None, None]),
}

# Pressure levels — GRIB values are [levels, lat, lon]
# Aurora expects [B, T, levels, H, W]
levels = np.array([
    1000, 925, 850, 700, 600,
    500, 400, 300, 250, 200,
    150, 100, 50
])

atmos_vars = {
    "t": torch.from_numpy(pl_ds["t"].sel(isobaricInhPa=levels).values[None, None]),
    "u": torch.from_numpy(pl_ds["u"].sel(isobaricInhPa=levels).values[None, None]),
    "v": torch.from_numpy(pl_ds["v"].sel(isobaricInhPa=levels).values[None, None]),
    "q": torch.from_numpy(pl_ds["q"].sel(isobaricInhPa=levels).values[None, None]),
    "z": torch.from_numpy(pl_ds["z"].sel(isobaricInhPa=levels).values[None, None]),
}

# Metadata
metadata = Metadata(
    lat=torch.from_numpy(pl_ds.latitude.values),
    lon=torch.from_numpy(pl_ds.longitude.values),
    time=(np.datetime64("2024-03-02T00:00"),),
    atmos_levels=levels,
)

# Batch
batch = Batch(
    surf_vars=surf_vars,
    static_vars={k: torch.from_numpy(v) for k, v in static_vars.items()},
    atmos_vars=atmos_vars,
    metadata=metadata,
)

# Verify shapes
print("Batch shapes:")
for k, v in batch.surf_vars.items():
    print(f"  surf  {k}: {v.shape}")  # expect [1, 1, 1801, 3600]
for k, v in batch.atmos_vars.items():
    print(f"  atmos {k}: {v.shape}")  # expect [1, 1, 13, 1801, 3600]
B, T = next(iter(batch.surf_vars.values())).shape[:2]
print(f"\n→ B={B}, T={T} (should be B=1, T=1)")

In [0]:
# ------------------------------
# Load model + run inference (SINGLE STEP)
# ------------------------------
print("Running Aurora inference (single step)...")

model.eval()
model = model.to("cpu")

with torch.inference_mode():
    preds = [p.to("cpu") for p in rollout(model, batch, steps=1)]
    torch.cuda.empty_cache()


In [0]:
# ------------------------------
# Save output (single forecast)
# ------------------------------
print("Saving forecast...")

batch_step = preds[0]

# Forecast time = +6 hours from input
step_time = np.datetime64(metadata.time[0]) + np.timedelta64(6, "h")

lat = batch_step.metadata.lat.cpu().numpy()
lon = lon_360_to_180(batch_step.metadata.lon.cpu().numpy())

# Atmospheric variables
atm_data = {
    v: (("isobaricInhPa", "latitude", "longitude"),
        np.squeeze(t.cpu().numpy()).astype(np.float32))
    for v, t in batch_step.atmos_vars.items()
}

# Surface variables
surf_data = {
    v: (("latitude", "longitude"),
        np.squeeze(t.cpu().numpy()).astype(np.float32))
    for v, t in batch_step.surf_vars.items()
}

# Static variables
static_data = {}
for v, t in batch_step.static_vars.items():
    var_name = "z_sfc" if v == "z" else v
    static_data[var_name] = (
        ("latitude", "longitude"),
        t.cpu().numpy().astype(np.float32)
    )

# Combine all variables
combined_vars = {**atm_data, **surf_data, **static_data}

# Create dataset
ds = xr.Dataset(
    data_vars=combined_vars,
    coords={
        "time": ("time", [step_time]),
        "valid_time": ("time", [step_time]),
        "isobaricInhPa": ("isobaricInhPa", levels),
        "latitude": ("latitude", lat),
        "longitude": ("longitude", lon),
    },
)

# Save file
out_path = os.path.join(output_dir, "aurora_forecast_single.nc")
ds.to_netcdf(out_path)

print("✓ Saved forecast: aurora_forecast_single.nc")
print("\nDone!")